# connections-rl — GRPO on a free Colab T4

Runtime → Change runtime type → **T4 GPU**.

Pipeline: clone → build data → (optionally) SFT warm start → GRPO → quick val eval.
Checkpoints save every 50 steps to Drive so a session timeout costs nothing.

In [ ]:
!nvidia-smi

In [ ]:
# Mount Drive for durable checkpoints
from google.colab import drive
drive.mount('/content/drive')
ARTIFACTS = '/content/drive/MyDrive/connections-rl-artifacts'
!mkdir -p $ARTIFACTS

In [ ]:
!git clone https://github.com/jacksonmlukas/connections-rl.git
%cd connections-rl
!pip install -q -e ".[train]"
# Colab preinstalls an old torchao that newer peft rejects; we don't use it.
!pip uninstall -q -y torchao
!git clone --depth 1 https://github.com/jacksonmlukas/gvc-local.git /content/gvc-local

In [ ]:
import os
os.environ['CONNECTIONS_PUZZLES'] = '/content/gvc-local/data/puzzles/tagged_connections.json'
!python -m connections_rl.data.build --out data/splits

In [ ]:
import wandb
wandb.login()  # paste your API key

## SFT warm start (skip if you already have artifacts/sft on Drive)

In [ ]:
!sed -i "s|output_dir: artifacts/sft|output_dir: $ARTIFACTS/sft|" configs/train/sft.yaml
!python -m connections_rl.train.sft --config configs/train/sft.yaml

## GRPO

In [ ]:
!sed -i "s|init_adapter: artifacts/sft|init_adapter: $ARTIFACTS/sft|" configs/train/grpo.yaml
!sed -i "s|output_dir: artifacts/grpo|output_dir: $ARTIFACTS/grpo|" configs/train/grpo.yaml
!python -m connections_rl.train.grpo --config configs/train/grpo.yaml

## Quick val check

Serve the adapter with vLLM in a separate cell/session, or push to HF Hub and eval from anywhere:
```
python -m connections_rl.eval.run --config configs/eval/default.yaml
```